In [1]:
# =============================================================================
# STEP 5b - DOES NOVELTY MATTER, OR ONLY WHERE THE SCORES SIT?
#
# Step 5 showed that warezmaster carries 92% of the NSL dose-response, and that my
# composition-adjusted regression was the wrong specification: composition is the
# MECHANISM the ladder operates through, not a confounder, so conditioning on it
# blocks the causal path and yields a coefficient with no causal reading.
#
# The answerable question is the reverse one. Among subtypes the source has ALREADY
# SEEN, coverage ranges from 0.007 (guess_passwd, 11 source points) to 0.243
# (warezmaster, 3 source points). So an evaluation set built entirely from seen
# subtypes can be made to swing coverage by changing their ratio alone, with the
# support shift pinned at exactly zero.
#
#   EXPERIMENT A (placebo ladder): vary the guess_passwd : warezmaster ratio across
#   five rungs, S_sup = 0 throughout, everything else held identical. If the swing
#   matches or exceeds the novelty ladder's 0.143 -> 0.030, then novelty is not the
#   operative variable and score position is, shown by construction.
#
#   EXPERIMENT B (invariance): is each subtype's OWN coverage constant across the
#   novelty rungs? If so, the class-level dose-response is exactly the mixture-
#   weighted average of fixed per-subtype coverages, which is what the mechanism
#   predicts and turns the LOSO result into a decomposition rather than a weakness.
#
# This can fail. If the placebo ladder produces a much SMALLER swing, novelty is
# doing independent work and the mechanism claim is too strong.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
from conformal import conformal_q
import numpy as np, pandas as pd
from scipy import stats
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY; FOCAL='R2L'
def dseed(*p): return int(hashlib.sha256('|'.join(map(str,p)).encode()).hexdigest(),16)%(2**32)
print('ready | alpha', ALPHA)


Mounted at /content/drive
ready | alpha 0.05


In [2]:
# =============================================================================
# Cell 2 - data, and the placebo ladder construction.
# The non-focal portion of each evaluation set uses IDENTICAL indices across rungs
# within a realisation, so only the R2L composition can move the result.
# =============================================================================
def aps_mid(P):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    ss=cum-0.5*sp; out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out
def aps_rand(P,rng):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    U=rng.random(len(P))[:,None]; ss=cum-(1-U)*sp
    out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

CL=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CL)}; FIDX=c2i[FOCAL]
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
src=tr[tr.partition=='source_cal_pool'].reset_index(drop=True)
y_sp=src['label'].map(c2i).to_numpy(); sub_sp=src['subtype'].to_numpy()
y_te=te['label'].map(c2i).to_numpy(); sub_te=te['subtype'].to_numpy()
seen=set(pd.Series(sub_sp[y_sp==FIDX]).unique())
man=json.loads((RD/'ladder_manifest_nslkdd.json').read_text())
QUOTA=man['quota_per_class']; NEVAL=man['d_eval_size']; NREAL=man['n_realizations']
print('focal quota', QUOTA[FOCAL], '| eval size', NEVAL, '| realizations', NREAL)
print('R2L subtypes SEEN in the source pool:', sorted(seen))

# the two seen subtypes with enough target mass to build a ladder from
A_SUB, B_SUB = 'warezmaster', 'guess_passwd'
idx_A=np.where((y_te==FIDX)&(sub_te==A_SUB))[0]
idx_B=np.where((y_te==FIDX)&(sub_te==B_SUB))[0]
print(f'  {A_SUB}: {len(idx_A)} target instances, in source: {A_SUB in seen}')
print(f'  {B_SUB}: {len(idx_B)} target instances, in source: {B_SUB in seen}')
assert A_SUB in seen and B_SUB in seen, 'placebo ladder must use only SEEN subtypes'
assert len(idx_A)>=QUOTA[FOCAL] and len(idx_B)>=QUOTA[FOCAL], 'insufficient mass for a full swing'

PL_RUNGS=[0.00,0.25,0.50,0.75,1.00]      # fraction of the focal quota drawn from B (guess_passwd)
nonfocal={c: np.where(y_te==c2i[c])[0] for c in CL if c!=FOCAL}
print(f'\nplacebo ladder: {PL_RUNGS} = fraction {B_SUB}, remainder {A_SUB}')
print('S_sup is 0.000 at every rung by construction, since both subtypes are in the source')


focal quota 299 | eval size 2340 | realizations 20
R2L subtypes SEEN in the source pool: ['ftp_write', 'guess_passwd', 'imap', 'multihop', 'warezclient', 'warezmaster']
  warezmaster: 944 target instances, in source: True
  guess_passwd: 1231 target instances, in source: True

placebo ladder: [0.0, 0.25, 0.5, 0.75, 1.0] = fraction guess_passwd, remainder warezmaster
S_sup is 0.000 at every rung by construction, since both subtypes are in the source


In [3]:
# =============================================================================
# Cell 3 - EXPERIMENT A. Run the placebo ladder over the full model panel.
# =============================================================================
rows=[]
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    d=np.load(f); P_sp=d['S_pool'].astype(np.float64); P_te=d['target'].astype(np.float64)
    for j in range(NREAL):
        rj=np.random.default_rng(dseed('placebo-nonfocal',j))
        # non-focal indices: drawn ONCE per realisation and reused at every rung
        fixed=[]
        for c,pool in nonfocal.items():
            q=min(QUOTA[c], len(pool))
            if q>0: fixed.append(rj.choice(pool, q, replace=False))
        fixed=np.concatenate(fixed) if fixed else np.array([],dtype=int)
        for rg in PL_RUNGS:
            nB=int(round(rg*QUOTA[FOCAL])); nA=QUOTA[FOCAL]-nB
            rr=np.random.default_rng(dseed('placebo',j,rg))
            pick=np.concatenate([rr.choice(idx_A,nA,replace=False),
                                 rr.choice(idx_B,nB,replace=False), fixed])
            rng=np.random.default_rng(dseed('placebo-score',arch,seed,j,rg))
            S_sp=aps_rand(P_sp, np.random.default_rng(dseed('placebo-src',arch,seed,j,rg)))
            S_ev=aps_rand(P_te[pick], rng)
            tc=S_sp[np.arange(len(y_sp)),y_sp]
            q_,_=conformal_q(tc[y_sp==FIDX], ALPHA)
            if not np.isfinite(q_): continue
            ye=y_te[pick]; fm=ye==FIDX
            cov=float((S_ev[fm,FIDX]<=q_).mean())
            # support shift, measured not assumed
            s_sup=float(np.isin(sub_te[pick][fm], list(seen), invert=True).mean())
            # movement, deterministic scores for reproducibility
            Sm_sp=aps_mid(P_sp); Sm_ev=aps_mid(P_te[pick])
            ks=float(stats.ks_2samp(Sm_sp[y_sp==FIDX,FIDX], Sm_ev[fm,FIDX]).statistic)
            rows.append({'ladder':'placebo_seen_only','rung':float(rg),'realization':j,
                         'arch':arch,'seed':seed,'coverage':cov,'S_sup':s_sup,'score_KS':ks,
                         'n_focal':int(fm.sum())})
PL=pd.DataFrame(rows)
g=PL.groupby('rung').agg(coverage=('coverage','mean'), S_sup=('S_sup','mean'),
                         score_KS=('score_KS','mean')).round(4)
print(f'EXPERIMENT A - placebo ladder ({B_SUB} fraction), all subtypes SEEN')
print(g.to_string())
assert PL.S_sup.max()<1e-9, f'support shift is not zero: max {PL.S_sup.max()}'
print('\n  S_sup = 0.000 at every rung, verified not assumed')
swing_pl=float(g.coverage.max()-g.coverage.min())
print(f'  coverage swing: {g.coverage.max():.4f} -> {g.coverage.min():.4f}  = {swing_pl:.4f}')


EXPERIMENT A - placebo ladder (guess_passwd fraction), all subtypes SEEN
      coverage  S_sup  score_KS
rung                           
0.00    0.3076    0.0    0.8005
0.25    0.2331    0.0    0.8364
0.50    0.1614    0.0    0.8729
0.75    0.0814    0.0    0.9144
1.00    0.0097    0.0    0.9716

  S_sup = 0.000 at every rung, verified not assumed
  coverage swing: 0.3076 -> 0.0097  = 0.2979


In [4]:
# =============================================================================
# Cell 4 - the comparison, and EXPERIMENT B.
# =============================================================================
cov=pd.read_csv(RD/'coverage_primary_nslkdd.csv')
nov=cov[(cov['class']==FOCAL)&(np.isclose(cov.alpha,ALPHA))&(cov.protocol=='SHC')]
novg=nov.groupby('rung')['coverage'].mean()
swing_nov=float(novg.max()-novg.min())
print('THE COMPARISON')
print(f'  novelty ladder  : {novg.max():.4f} -> {novg.min():.4f}  swing {swing_nov:.4f}   '
      f'(S_sup 0.00 -> {float(pd.read_csv(RD/"ladder_shift_measures_nslkdd.csv").S_sup.max()):.2f})')
print(f'  placebo ladder  : {g.coverage.max():.4f} -> {g.coverage.min():.4f}  swing {swing_pl:.4f}   '
      f'(S_sup 0.00 throughout)')
print(f'  ratio placebo/novelty: {swing_pl/swing_nov:.2f}')
print()
if swing_pl >= swing_nov:
    print('  VERDICT: composition among ALREADY-SEEN subtypes moves coverage at least as much')
    print('  as the novelty manipulation, with zero support shift. Novelty is therefore not')
    print('  the operative variable; where a subtype\u2019s scores sit relative to the source')
    print('  quantile is. This is shown by construction rather than by correlation.')
elif swing_pl >= 0.5*swing_nov:
    print('  VERDICT: seen-only composition produces a substantial but smaller swing.')
    print('  Novelty contributes something beyond score position; report both.')
else:
    print('  VERDICT: the placebo swing is small. Novelty IS doing independent work and the')
    print('  mechanism claim as stated is too strong. Report this against the paper.')

print('\nEXPERIMENT B - is each subtype\u2019s OWN coverage invariant across the novelty rungs?')
ST=pd.read_csv(RD/'nsl_subtype_cells.csv')
inv=[]
for s,gg in ST.groupby('subtype'):
    if gg.rung.nunique()<3 or gg.n_target.sum()<50: continue
    byr=gg.groupby('rung').apply(lambda d: float((d.coverage*d.n_target).sum()/d.n_target.sum()))
    sl,_,r,p,_=stats.linregress(byr.index.astype(float), byr.values)
    inv.append({'subtype':s,'mean_coverage':round(float(gg.coverage.mean()),4),
                'slope_vs_rung':round(float(sl),4),'r':round(float(r),3),'p':round(float(p),4),
                'range':round(float(byr.max()-byr.min()),4)})
INV=pd.DataFrame(inv).sort_values('mean_coverage',ascending=False)
print(INV.to_string(index=False))
print(f'\n  largest within-subtype range across rungs: {INV["range"].max():.4f}')
print(f'  class-level swing being explained:        {swing_nov:.4f}')
if INV['range'].max() < 0.5*swing_nov:
    print('  => per-subtype coverage is broadly invariant, so the class-level dose-response is')
    print('     the mixture-weighted average of fixed per-subtype coverages. The LOSO result is')
    print('     then a DECOMPOSITION of the effect, not a fragility of it.')
else:
    print('  => per-subtype coverage itself drifts with rung, so something beyond composition')
    print('     is operating and the mixture account is incomplete.')


THE COMPARISON
  novelty ladder  : 0.1431 -> 0.0298  swing 0.1133   (S_sup 0.00 -> 0.46)
  placebo ladder  : 0.3076 -> 0.0097  swing 0.2979   (S_sup 0.00 throughout)
  ratio placebo/novelty: 2.63

  VERDICT: composition among ALREADY-SEEN subtypes moves coverage at least as much
  as the novelty manipulation, with zero support shift. Novelty is therefore not
  the operative variable; where a subtype’s scores sit relative to the source
  quantile is. This is shown by construction rather than by correlation.

EXPERIMENT B - is each subtype’s OWN coverage invariant across the novelty rungs?
      subtype  mean_coverage  slope_vs_rung      r      p  range
  warezmaster         0.2431         0.0013  0.166 0.7894 0.0054
     multihop         0.0373        -0.1046 -0.965 0.1691 0.0418
     sendmail         0.0248        -0.0298 -0.471 0.5295 0.0394
 guess_passwd         0.0072        -0.0020 -0.857 0.0632 0.0016
        named         0.0047         0.0020  0.548 0.4516 0.0022
   httptunnel  

/tmp/ipykernel_3586/3191966600.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  byr=gg.groupby('rung').apply(lambda d: float((d.coverage*d.n_target).sum()/d.n_target.sum()))
/tmp/ipykernel_3586/3191966600.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  byr=gg.groupby('rung').apply(lambda d: float((d.coverage*d.n_target).sum()/d.n_target.sum()))
/tmp/ipykernel_3586/3191966600.py:32: DeprecationWarni

In [5]:
# =============================================================================
# Cell 5 - figure, save, commit.
# =============================================================================
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,
                     'axes.spines.right':False,'figure.dpi':300})
fig,axs=plt.subplots(1,2,figsize=(9.8,4.2))
axs[0].plot(novg.index.astype(float), novg.values, marker='o', ms=5, color='#2f4b7c',
            label=f'novelty ladder (S_sup 0 to 0.45)')
axs[0].plot(np.linspace(0,0.8,len(g)), g.coverage.values, marker='D', ms=5, color='#b3261e',
            label='placebo ladder, seen subtypes only (S_sup = 0)')
axs[0].axhline(1-ALPHA, color='#666', ls=':', lw=1, label='nominal 0.95')
axs[0].set_xlabel('ladder position (rescaled to a common axis)')
axs[0].set_ylabel('focal (R2L) coverage under SHC')
axs[0].set_title('(a) novelty is not required to move coverage')
axs[0].legend(fontsize=7.5, frameon=False)
sc=INV.dropna(subset=['mean_coverage'])
axs[1].barh(sc.subtype, sc['range'], color='#4a5a2f')
axs[1].axvline(swing_nov, color='#b3261e', ls='--', lw=1, label='class-level swing')
axs[1].set_xlabel('within-subtype coverage range across rungs')
axs[1].set_title('(b) per-subtype coverage is broadly invariant')
axs[1].legend(fontsize=7.5, frameon=False)
fig.tight_layout(); fig.savefig(RD/'placebo_ladder.png',bbox_inches='tight',facecolor='white')
print('figure saved: placebo_ladder.png')

PL.to_csv(RD/'placebo_ladder_cells.csv', index=False)
g.to_csv(RD/'placebo_ladder_curve.csv')
INV.to_csv(RD/'subtype_coverage_invariance.csv', index=False)
(RD/'placebo_ladder_verdict.json').write_text(json.dumps({
 'design':'R2L evaluation sets built ONLY from subtypes present in the source '
          f'({A_SUB}, {B_SUB}); non-focal indices held identical across rungs within a '
          'realisation so only focal composition varies',
 'S_sup_max_observed':float(PL.S_sup.max()),
 'placebo_swing':swing_pl,'novelty_swing':swing_nov,'ratio':swing_pl/swing_nov,
 'placebo_curve':g.reset_index().to_dict('records'),
 'per_subtype_invariance':INV.to_dict('records'),
 'note':'the composition-adjusted regression in step 5 conditioned on mediators and is '
        'withdrawn; this design answers the question it could not'}, indent=2, default=str))
print('saved placebo cells, curve, invariance table and verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','step 5b: seen-only placebo ladder and per-subtype invariance; tests whether novelty or score position drives the dose-response')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


figure saved: placebo_ladder.png
saved placebo cells, curve, invariance table and verdict
[main 23e099a] step 5b: seen-only placebo ladder and per-subtype invariance; tests whether novelty or score position drives the dose-response
 7 files changed, 3124 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/41_placebo_ladder.ipynb
 create mode 100644 reports/placebo_ladder.png
 create mode 100644 reports/placebo_ladder_cells.csv
 create mode 100644 reports/placebo_ladder_curve.csv
 create mode 100644 reports/placebo_ladder_verdict.json
 create mode 100644 reports/subtype_coverage_invariance.csv
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   9a55394..23e099a  main -> main
23e099a step 5b: seen-only placebo ladder and per-subtype invariance; tests whether novelty or score position drives the dose-response
9a55394 step 5: NSL ladder sensitivity; tests whether novelty, source support or score movement predic